# Advanced `OrderedDict` Problems — With Complete Solutions

This notebook develops advanced, practical mastery of `collections.OrderedDict`.

It is based on the supplied lesson material covering:

- insertion order,
- `popitem(last=True/False)`,
- `move_to_end(key, last=True/False)`,
- order-sensitive equality between `OrderedDict` objects,
- order-insensitive equality when comparing with a normal `dict`,
- stack/queue-style behavior,
- and performance tradeoffs versus `deque`.

## Modern Python note

In modern Python, plain `dict` preserves insertion order. Therefore, use `OrderedDict`
when you specifically need **order-manipulation semantics** such as:

- efficiently moving an existing key to the front/end,
- efficiently popping from either end by key/value pair,
- order-sensitive equality between two ordered mappings,
- or algorithms whose behavior explicitly depends on changing key order.

For ordinary "remember insertion order" use cases, prefer a regular `dict`.

---

## How to use this notebook

Each section follows this pattern:

1. **Problem**
2. **Requirements / constraints**
3. **Solution**
4. **Tests**
5. **Complexity / best-practice notes**
6. **Extensions**

Try solving each problem before revealing/running the solution cells.

In [1]:
from collections import OrderedDict, deque
from collections.abc import MutableMapping
from dataclasses import dataclass
from timeit import repeat
from typing import Any, Iterable, Iterator, Hashable
import random
import statistics

## Warm-up: Core behavior refresher

In [2]:
od = OrderedDict([
    ("alpha", 10),
    ("beta", 20),
    ("gamma", 30),
])

print("Original:", od)

od.move_to_end("alpha")
print("alpha -> end:", od)

od.move_to_end("gamma", last=False)
print("gamma -> front:", od)

last_item = od.popitem()
first_item = od.popitem(last=False)

print("Popped last:", last_item)
print("Popped first:", first_item)
print("Remaining:", od)

Original: OrderedDict({'alpha': 10, 'beta': 20, 'gamma': 30})
alpha -> end: OrderedDict({'beta': 20, 'gamma': 30, 'alpha': 10})
gamma -> front: OrderedDict({'gamma': 30, 'beta': 20, 'alpha': 10})
Popped last: ('alpha', 10)
Popped first: ('gamma', 30)
Remaining: OrderedDict({'beta': 20})


### Important equality behavior

In [3]:
a = OrderedDict([("x", 1), ("y", 2)])
b = OrderedDict([("y", 2), ("x", 1)])
plain = {"y": 2, "x": 1}

print("OrderedDict vs OrderedDict:", a == b)   # order matters
print("OrderedDict vs dict:", a == plain)      # mapping contents matter
print("dict vs OrderedDict:", plain == a)

OrderedDict vs OrderedDict: False
OrderedDict vs dict: True
dict vs OrderedDict: True


# Problem 1 — Build an LRU Cache

Implement a fixed-capacity **Least Recently Used (LRU)** cache.

## Requirements

Create a class `LRUCache` with:

- `capacity > 0`,
- `get(key, default=None)`,
- `put(key, value)`,
- `__contains__(key)`,
- `__len__()`,
- `items_lru_to_mru()`.

Rules:

- A successful `get` makes the key the **most recently used**.
- Updating an existing key also makes it most recently used.
- When inserting beyond capacity, evict the **least recently used** key.
- Keep operations near O(1).

## Solution 1

In [4]:
class LRUCache:
    def __init__(self, capacity: int):
        if not isinstance(capacity, int):
            raise TypeError("capacity must be an int")
        if capacity <= 0:
            raise ValueError("capacity must be > 0")
        self.capacity = capacity
        self._data = OrderedDict()

    def get(self, key, default=None):
        if key not in self._data:
            return default

        value = self._data[key]
        self._data.move_to_end(key)  # mark as most recently used
        return value

    def put(self, key, value):
        if key in self._data:
            self._data[key] = value
            self._data.move_to_end(key)
            return None

        self._data[key] = value

        if len(self._data) > self.capacity:
            # first item = least recently used
            evicted_key, evicted_value = self._data.popitem(last=False)
            return evicted_key, evicted_value

        return None

    def __contains__(self, key):
        return key in self._data

    def __len__(self):
        return len(self._data)

    def items_lru_to_mru(self):
        return list(self._data.items())

    def __repr__(self):
        return f"LRUCache(capacity={self.capacity}, data={self._data!r})"

In [5]:
cache = LRUCache(3)

assert cache.put("a", 1) is None
assert cache.put("b", 2) is None
assert cache.put("c", 3) is None
assert cache.items_lru_to_mru() == [("a", 1), ("b", 2), ("c", 3)]

assert cache.get("a") == 1
assert cache.items_lru_to_mru() == [("b", 2), ("c", 3), ("a", 1)]

evicted = cache.put("d", 4)
assert evicted == ("b", 2)
assert cache.items_lru_to_mru() == [("c", 3), ("a", 1), ("d", 4)]

cache.put("a", 100)
assert cache.items_lru_to_mru() == [("c", 3), ("d", 4), ("a", 100)]

assert cache.get("missing", -1) == -1

print(cache)
print("All LRU tests passed.")

LRUCache(capacity=3, data=OrderedDict({'c': 3, 'd': 4, 'a': 100}))
All LRU tests passed.


### Why `OrderedDict` fits

`move_to_end(key)` changes recency without rebuilding the mapping, while
`popitem(last=False)` removes the least-recent item directly.

**Expected complexity:** average O(1) lookup, update, move, and eviction.

### Best practice

Do not implement LRU behavior by repeatedly converting dictionaries to lists or
sorting timestamps; that usually adds unnecessary O(n) work.

# Problem 2 — LRU Cache With Hit/Miss/Eviction Statistics

Extend the previous design so the cache records:

- hits,
- misses,
- insertions,
- updates,
- evictions,
- hit rate.

Also add `peek(key)` which returns a value **without changing recency**.

## Solution 2

In [6]:
@dataclass
class CacheStats:
    hits: int = 0
    misses: int = 0
    insertions: int = 0
    updates: int = 0
    evictions: int = 0

    @property
    def requests(self):
        return self.hits + self.misses

    @property
    def hit_rate(self):
        return self.hits / self.requests if self.requests else 0.0


class InstrumentedLRUCache:
    def __init__(self, capacity: int):
        if capacity <= 0:
            raise ValueError("capacity must be > 0")
        self.capacity = capacity
        self._data = OrderedDict()
        self.stats = CacheStats()

    def get(self, key, default=None):
        if key not in self._data:
            self.stats.misses += 1
            return default

        self.stats.hits += 1
        self._data.move_to_end(key)
        return self._data[key]

    def peek(self, key, default=None):
        return self._data.get(key, default)

    def put(self, key, value):
        if key in self._data:
            self.stats.updates += 1
            self._data[key] = value
            self._data.move_to_end(key)
            return None

        self.stats.insertions += 1
        self._data[key] = value

        if len(self._data) > self.capacity:
            self.stats.evictions += 1
            return self._data.popitem(last=False)

        return None

    def order(self):
        return list(self._data)

In [7]:
cache = InstrumentedLRUCache(2)

cache.put("x", 10)
cache.put("y", 20)

# peek must NOT change order
before = cache.order()
assert cache.peek("x") == 10
assert cache.order() == before

assert cache.get("x") == 10
assert cache.get("missing") is None

cache.put("z", 30)
cache.put("x", 11)

assert cache.stats.hits == 1
assert cache.stats.misses == 1
assert cache.stats.insertions == 3
assert cache.stats.updates == 1
assert cache.stats.evictions == 1
assert cache.stats.hit_rate == 0.5

print(cache.stats)
print("Current LRU -> MRU:", cache.order())

CacheStats(hits=1, misses=1, insertions=3, updates=1, evictions=1)
Current LRU -> MRU: ['z', 'x']


# Problem 3 — MRU Cache

An **MRU cache** evicts the **most recently used** item instead of the least recently used.

Implement `MRUCache` with the same recency rules as the LRU cache, but when capacity
is exceeded, evict the current most-recent item *before* the newly inserted item
becomes the survivor.

Example with capacity 3:

`A, B, C`, then access `A` -> order becomes `B, C, A`.

Insert `D`. The old MRU `A` should be evicted, producing `B, C, D`.

## Solution 3

In [8]:
class MRUCache:
    def __init__(self, capacity: int):
        if capacity <= 0:
            raise ValueError("capacity must be > 0")
        self.capacity = capacity
        self._data = OrderedDict()

    def get(self, key, default=None):
        if key not in self._data:
            return default
        self._data.move_to_end(key)
        return self._data[key]

    def put(self, key, value):
        if key in self._data:
            self._data[key] = value
            self._data.move_to_end(key)
            return None

        evicted = None

        # If full, evict the current MRU BEFORE adding the new key.
        if len(self._data) >= self.capacity:
            evicted = self._data.popitem(last=True)

        self._data[key] = value
        return evicted

    def items(self):
        return list(self._data.items())

In [9]:
mru = MRUCache(3)
mru.put("A", 1)
mru.put("B", 2)
mru.put("C", 3)

assert mru.get("A") == 1
assert [k for k, _ in mru.items()] == ["B", "C", "A"]

assert mru.put("D", 4) == ("A", 1)
assert [k for k, _ in mru.items()] == ["B", "C", "D"]

print(mru.items())

[('B', 2), ('C', 3), ('D', 4)]


# Problem 4 — FIFO Queue With O(1) Membership Checks

A normal `deque` is excellent for queue operations but membership tests are O(n).

Build a queue for **unique IDs** where you need:

- append a new ID,
- pop the oldest ID,
- check membership efficiently,
- reject duplicates.

Use an `OrderedDict` as an ordered set.

## Solution 4

In [10]:
class UniqueFIFO:
    def __init__(self):
        self._data = OrderedDict()

    def add(self, item):
        if item in self._data:
            return False
        self._data[item] = None
        return True

    def pop_oldest(self):
        if not self._data:
            raise IndexError("pop from empty UniqueFIFO")
        key, _ = self._data.popitem(last=False)
        return key

    def __contains__(self, item):
        return item in self._data

    def __len__(self):
        return len(self._data)

    def __iter__(self):
        return iter(self._data)

In [11]:
q = UniqueFIFO()

assert q.add("job-101") is True
assert q.add("job-102") is True
assert q.add("job-101") is False

assert "job-102" in q
assert q.pop_oldest() == "job-101"
assert list(q) == ["job-102"]

print("Queue:", list(q))

Queue: ['job-102']


### Design tradeoff

Use `deque` when you mainly need fast append/pop operations.

Use an ordered mapping/set-like structure when you also need fast keyed lookup,
deduplication, or value association.

# Problem 5 — Move-to-Front Priority List

Implement a list of `(task_id, payload)` pairs with these rules:

- New tasks go to the end.
- `promote(task_id)` moves a task to the front.
- `demote(task_id)` moves a task to the end.
- `pop_next()` removes from the front.
- Updating a payload must **not** alter position.

## Solution 5

In [12]:
class TaskPriorityList:
    def __init__(self):
        self._tasks = OrderedDict()

    def add_or_update(self, task_id, payload):
        self._tasks[task_id] = payload

    def promote(self, task_id):
        self._tasks.move_to_end(task_id, last=False)

    def demote(self, task_id):
        self._tasks.move_to_end(task_id, last=True)

    def pop_next(self):
        if not self._tasks:
            raise IndexError("no tasks available")
        return self._tasks.popitem(last=False)

    def snapshot(self):
        return list(self._tasks.items())

In [13]:
tasks = TaskPriorityList()
tasks.add_or_update("T1", {"name": "backup"})
tasks.add_or_update("T2", {"name": "report"})
tasks.add_or_update("T3", {"name": "deploy"})

tasks.promote("T3")
assert [k for k, _ in tasks.snapshot()] == ["T3", "T1", "T2"]

# Updating should keep T1 in place.
tasks.add_or_update("T1", {"name": "backup", "retries": 1})
assert [k for k, _ in tasks.snapshot()] == ["T3", "T1", "T2"]

tasks.demote("T3")
assert [k for k, _ in tasks.snapshot()] == ["T1", "T2", "T3"]

print("Next:", tasks.pop_next())
print("Remaining:", tasks.snapshot())

Next: ('T1', {'name': 'backup', 'retries': 1})
Remaining: [('T2', {'name': 'report'}), ('T3', {'name': 'deploy'})]


# Problem 6 — Stable Deduplication With "First Wins" and "Last Wins"

Given an iterable, produce unique values while preserving a meaningful order.

Implement:

1. `dedupe_first_wins(items)`  
   Keep the position of the first appearance.

2. `dedupe_last_wins(items)`  
   Keep values ordered according to their final appearance.

Example:

`["a", "b", "a", "c", "b"]`

- first wins -> `["a", "b", "c"]`
- last wins -> `["a", "c", "b"]`

## Solution 6

In [14]:
def dedupe_first_wins(items):
    seen = OrderedDict()
    for item in items:
        seen.setdefault(item, None)
    return list(seen)


def dedupe_last_wins(items):
    seen = OrderedDict()
    for item in items:
        if item in seen:
            seen.move_to_end(item)
        else:
            seen[item] = None
    return list(seen)

In [15]:
data = ["a", "b", "a", "c", "b"]

assert dedupe_first_wins(data) == ["a", "b", "c"]
assert dedupe_last_wins(data) == ["a", "c", "b"]

print("first wins:", dedupe_first_wins(data))
print("last wins :", dedupe_last_wins(data))

first wins: ['a', 'b', 'c']
last wins : ['a', 'c', 'b']


### Modern alternative

For *first-wins* deduplication in modern Python, `list(dict.fromkeys(items))` is concise
and idiomatic.

`OrderedDict` becomes more useful when you explicitly need to reposition previously
seen elements, as in the last-wins ordering above.

# Problem 7 — Order-Sensitive Configuration Comparison

Suppose pipeline configurations are mappings where execution order matters.

Create:

- `same_mapping(a, b)` — compares only key/value contents.
- `same_ordered_pipeline(a, b)` — compares both contents and order.
- `first_order_difference(a, b)` — returns the first index where ordered items differ.

The last function should return `None` if no difference exists.

## Solution 7

In [16]:
def same_mapping(a, b):
    return dict(a) == dict(b)


def same_ordered_pipeline(a, b):
    return list(a.items()) == list(b.items())


def first_order_difference(a, b):
    a_items = list(a.items())
    b_items = list(b.items())

    limit = max(len(a_items), len(b_items))

    for i in range(limit):
        left = a_items[i] if i < len(a_items) else None
        right = b_items[i] if i < len(b_items) else None

        if left != right:
            return i, left, right

    return None

In [17]:
p1 = OrderedDict([
    ("extract", "v1"),
    ("transform", "v2"),
    ("load", "v1"),
])

p2 = OrderedDict([
    ("transform", "v2"),
    ("extract", "v1"),
    ("load", "v1"),
])

assert same_mapping(p1, p2) is True
assert same_ordered_pipeline(p1, p2) is False
assert first_order_difference(p1, p2) == (
    0,
    ("extract", "v1"),
    ("transform", "v2"),
)

print(first_order_difference(p1, p2))

(0, ('extract', 'v1'), ('transform', 'v2'))


# Problem 8 — Reconcile an Existing Order Against a Desired Order

You have an existing `OrderedDict` and a sequence of desired keys.

Reorder only keys that already exist.

Rules:

- Keys listed in `desired_order` should appear first, in that order.
- Unknown desired keys should be ignored.
- Existing keys not mentioned in `desired_order` should remain afterward in their
  previous relative order.
- Do not rebuild values manually.

## Solution 8

In [18]:
def reconcile_order(mapping: OrderedDict, desired_order):
    desired_seen = set()

    # Move desired keys to the end in desired sequence.
    for key in desired_order:
        if key in mapping and key not in desired_seen:
            mapping.move_to_end(key)
            desired_seen.add(key)

    # At this point, desired keys are at the end.
    # Rotate them to the front while preserving desired order.
    for key in reversed(list(desired_seen)):
        # This route would lose desired sequence because sets are unordered.
        # So we intentionally do NOT use it.
        pass

    # Correct approach: replay the desired sequence in reverse to the front.
    for key in reversed(list(dict.fromkeys(desired_order))):
        if key in mapping:
            mapping.move_to_end(key, last=False)

    return mapping

In [19]:
od = OrderedDict([
    ("a", 1),
    ("b", 2),
    ("c", 3),
    ("d", 4),
    ("e", 5),
])

reconcile_order(od, ["d", "b", "missing", "d"])

assert list(od) == ["d", "b", "a", "c", "e"]
print(od)

OrderedDict({'d': 4, 'b': 2, 'a': 1, 'c': 3, 'e': 5})


### Cleaner implementation

The previous version intentionally exposes a common trap: using a `set` when order matters.

Here is the preferred version.

In [20]:
def reconcile_order(mapping: OrderedDict, desired_order):
    desired_unique = list(dict.fromkeys(desired_order))

    for key in reversed(desired_unique):
        if key in mapping:
            mapping.move_to_end(key, last=False)

    return mapping

In [21]:
od = OrderedDict((ch, ord(ch)) for ch in "abcde")
reconcile_order(od, ["d", "b", "missing", "d"])

assert list(od) == ["d", "b", "a", "c", "e"]
print(list(od.items()))

[('d', 100), ('b', 98), ('a', 97), ('c', 99), ('e', 101)]


# Problem 9 — Bounded Recent-Event Index

Implement a structure that stores at most `capacity` unique event IDs.

Behavior:

- `record(event_id, payload)` inserts or updates.
- Recording an existing event marks it newest.
- When full, the oldest event is removed.
- `oldest()` and `newest()` return `(id, payload)` without removing.
- `discard(event_id)` removes safely and returns whether removal occurred.

## Solution 9

In [22]:
class RecentEventIndex:
    def __init__(self, capacity):
        if capacity <= 0:
            raise ValueError("capacity must be > 0")
        self.capacity = capacity
        self._events = OrderedDict()

    def record(self, event_id, payload):
        self._events[event_id] = payload
        self._events.move_to_end(event_id)

        if len(self._events) > self.capacity:
            return self._events.popitem(last=False)

        return None

    def oldest(self):
        if not self._events:
            raise LookupError("index is empty")
        key = next(iter(self._events))
        return key, self._events[key]

    def newest(self):
        if not self._events:
            raise LookupError("index is empty")
        key = next(reversed(self._events))
        return key, self._events[key]

    def discard(self, event_id):
        try:
            del self._events[event_id]
            return True
        except KeyError:
            return False

    def __len__(self):
        return len(self._events)

In [23]:
events = RecentEventIndex(3)

events.record("e1", {"value": 10})
events.record("e2", {"value": 20})
events.record("e3", {"value": 30})

assert events.oldest()[0] == "e1"
assert events.newest()[0] == "e3"

events.record("e1", {"value": 11})
assert events.newest() == ("e1", {"value": 11})

evicted = events.record("e4", {"value": 40})
assert evicted[0] == "e2"

assert events.discard("missing") is False
assert events.discard("e3") is True

print("Oldest:", events.oldest())
print("Newest:", events.newest())

Oldest: ('e1', {'value': 11})
Newest: ('e4', {'value': 40})


# Problem 10 — Implement a TTL + LRU Cache

Now combine two eviction policies:

1. Expire entries after a fixed TTL measured in logical time.
2. Among non-expired entries, use LRU capacity eviction.

To keep the exercise deterministic, do not use wall-clock time.
Pass `now` explicitly into every operation.

Store each value as `(payload, expires_at)`.

## Solution 10

In [24]:
class TTLLRUCache:
    def __init__(self, capacity: int, ttl: float):
        if capacity <= 0:
            raise ValueError("capacity must be > 0")
        if ttl <= 0:
            raise ValueError("ttl must be > 0")

        self.capacity = capacity
        self.ttl = ttl
        self._data = OrderedDict()

    def _purge_expired(self, now):
        expired_keys = [
            key
            for key, (_, expires_at) in self._data.items()
            if expires_at <= now
        ]

        for key in expired_keys:
            del self._data[key]

        return expired_keys

    def put(self, key, value, now):
        self._purge_expired(now)

        expires_at = now + self.ttl
        self._data[key] = (value, expires_at)
        self._data.move_to_end(key)

        evicted = None
        if len(self._data) > self.capacity:
            evicted = self._data.popitem(last=False)

        return evicted

    def get(self, key, now, default=None):
        self._purge_expired(now)

        if key not in self._data:
            return default

        value, expires_at = self._data[key]
        self._data.move_to_end(key)
        return value

    def snapshot(self, now):
        self._purge_expired(now)
        return [
            (key, value, expires_at)
            for key, (value, expires_at) in self._data.items()
        ]

In [25]:
cache = TTLLRUCache(capacity=2, ttl=5)

cache.put("a", 1, now=0)
cache.put("b", 2, now=1)

assert cache.get("a", now=2) == 1
assert [x[0] for x in cache.snapshot(now=2)] == ["b", "a"]

# b expires at time 6; at 6 it is expired.
assert cache.get("b", now=6) is None

cache.put("c", 3, now=6)
print(cache.snapshot(now=6))

[('c', 3, 11)]


### Advanced discussion

The `_purge_expired` implementation scans all entries, so expiration cleanup is O(n).

For high-throughput production systems, common alternatives include:

- a min-heap keyed by expiration time,
- timing wheels,
- lazy expiration on access,
- periodic cleanup,
- or specialized cache libraries.

`OrderedDict` solves the recency dimension elegantly, but not every dimension of a
multi-policy cache should be forced into one data structure.

# Problem 11 — Detect Recency Corruption

You are given an LRU-like `OrderedDict` and an access log.

Each access should move the key to the end.

Write `replay_accesses(initial, accesses)` and return:

- the final ordered mapping,
- missing keys encountered,
- the final LRU -> MRU order.

Do not silently insert missing keys.

## Solution 11

In [26]:
def replay_accesses(initial, accesses):
    state = OrderedDict(initial)
    missing = []

    for key in accesses:
        if key not in state:
            missing.append(key)
            continue
        state.move_to_end(key)

    return state, missing, list(state)

In [27]:
initial = [("A", 10), ("B", 20), ("C", 30), ("D", 40)]
accesses = ["B", "A", "X", "C", "B"]

state, missing, order = replay_accesses(initial, accesses)

assert missing == ["X"]
assert order == ["D", "A", "C", "B"]

print("Final state:", state)
print("Missing:", missing)
print("LRU -> MRU:", order)

Final state: OrderedDict({'D': 40, 'A': 10, 'C': 30, 'B': 20})
Missing: ['X']
LRU -> MRU: ['D', 'A', 'C', 'B']


# Problem 12 — Ordered Merge With Conflict Policies

Merge multiple mappings while preserving order.

Implement:

```python
ordered_merge(*mappings, on_conflict="keep_position")
```

Conflict policies:

- `"keep_position"`:
  update the value but keep the key's original position.
- `"move_to_end"`:
  update the value and make the repeated key newest.
- `"error"`:
  raise `KeyError` if a key appears more than once.

## Solution 12

In [28]:
def ordered_merge(*mappings, on_conflict="keep_position"):
    valid = {"keep_position", "move_to_end", "error"}
    if on_conflict not in valid:
        raise ValueError(f"on_conflict must be one of {sorted(valid)}")

    result = OrderedDict()

    for mapping in mappings:
        for key, value in mapping.items():
            exists = key in result

            if exists and on_conflict == "error":
                raise KeyError(f"duplicate key: {key!r}")

            result[key] = value

            if exists and on_conflict == "move_to_end":
                result.move_to_end(key)

    return result

In [29]:
m1 = OrderedDict([("a", 1), ("b", 2)])
m2 = OrderedDict([("c", 3), ("a", 10)])

keep = ordered_merge(m1, m2, on_conflict="keep_position")
move = ordered_merge(m1, m2, on_conflict="move_to_end")

assert list(keep.items()) == [("a", 10), ("b", 2), ("c", 3)]
assert list(move.items()) == [("b", 2), ("c", 3), ("a", 10)]

print("keep_position:", keep)
print("move_to_end :", move)

keep_position: OrderedDict({'a': 10, 'b': 2, 'c': 3})
move_to_end : OrderedDict({'b': 2, 'c': 3, 'a': 10})


In [30]:
try:
    ordered_merge(m1, m2, on_conflict="error")
except KeyError as exc:
    print("Expected error:", exc)

Expected error: "duplicate key: 'a'"


# Problem 13 — Ordered Multiset of Latest Values

For each key, retain only its latest value, but order keys by the time of their
**latest update**.

Input:

```python
[
    ("a", 1),
    ("b", 2),
    ("a", 3),
    ("c", 4),
    ("b", 5),
]
```

Expected final order:

```python
[("a", 3), ("c", 4), ("b", 5)]
```

## Solution 13

In [31]:
def latest_update_order(pairs):
    result = OrderedDict()

    for key, value in pairs:
        result[key] = value
        result.move_to_end(key)

    return result

In [32]:
pairs = [
    ("a", 1),
    ("b", 2),
    ("a", 3),
    ("c", 4),
    ("b", 5),
]

result = latest_update_order(pairs)

assert list(result.items()) == [
    ("a", 3),
    ("c", 4),
    ("b", 5),
]

print(result)

OrderedDict({'a': 3, 'c': 4, 'b': 5})


# Problem 14 — Least-Recently-Changed Configuration Registry

Implement a registry where:

- inserting a new key makes it most recently changed,
- updating an existing key makes it most recently changed,
- reading does **not** change order,
- `pop_stalest()` removes the least recently changed item.

This differs subtly from LRU: reads do not affect recency.

## Solution 14

In [33]:
class ChangeRegistry:
    def __init__(self):
        self._data = OrderedDict()

    def set(self, key, value):
        self._data[key] = value
        self._data.move_to_end(key)

    def get(self, key, default=None):
        return self._data.get(key, default)

    def pop_stalest(self):
        if not self._data:
            raise KeyError("registry is empty")
        return self._data.popitem(last=False)

    def order(self):
        return list(self._data)

In [34]:
reg = ChangeRegistry()
reg.set("db", "v1")
reg.set("api", "v1")
reg.set("worker", "v1")

# Read must not change order.
assert reg.get("db") == "v1"
assert reg.order() == ["db", "api", "worker"]

reg.set("db", "v2")
assert reg.order() == ["api", "worker", "db"]

assert reg.pop_stalest() == ("api", "v1")

print(reg.order())

['worker', 'db']


# Problem 15 — Build an OrderedSet

Implement a small `OrderedSet` using `OrderedDict`.

Required operations:

- `add`
- `discard`
- `remove`
- membership
- iteration
- reverse iteration
- `pop(last=True)`
- `move_to_end`
- length

## Solution 15

In [35]:
class OrderedSet:
    _PRESENT = object()

    def __init__(self, iterable=()):
        self._data = OrderedDict()
        for item in iterable:
            self.add(item)

    def add(self, item):
        self._data[item] = self._PRESENT

    def discard(self, item):
        self._data.pop(item, None)

    def remove(self, item):
        del self._data[item]

    def pop(self, last=True):
        if not self._data:
            raise KeyError("set is empty")
        item, _ = self._data.popitem(last=last)
        return item

    def move_to_end(self, item, last=True):
        self._data.move_to_end(item, last=last)

    def __contains__(self, item):
        return item in self._data

    def __iter__(self):
        return iter(self._data)

    def __reversed__(self):
        return reversed(self._data)

    def __len__(self):
        return len(self._data)

    def __repr__(self):
        return f"OrderedSet({list(self._data)!r})"

In [36]:
s = OrderedSet(["a", "b", "c", "a"])

assert list(s) == ["a", "b", "c"]
assert list(reversed(s)) == ["c", "b", "a"]

s.move_to_end("a")
assert list(s) == ["b", "c", "a"]

s.move_to_end("c", last=False)
assert list(s) == ["c", "b", "a"]

assert s.pop(last=False) == "c"
assert "c" not in s

print(s)

OrderedSet(['b', 'a'])


# Problem 16 — Custom Mapping With Access-Order Iteration

Implement a `MutableMapping` whose iteration order reflects access recency.

Rules:

- setting a key moves it to the end,
- successful `__getitem__` moves it to the end,
- deleting removes it,
- iteration goes LRU -> MRU.

This exercise demonstrates how `OrderedDict` can serve as an internal engine for a
higher-level abstraction.

## Solution 16

In [37]:
class AccessOrderedMap(MutableMapping):
    def __init__(self, *args, **kwargs):
        self._data = OrderedDict()
        self.update(*args, **kwargs)

    def __getitem__(self, key):
        value = self._data[key]
        self._data.move_to_end(key)
        return value

    def __setitem__(self, key, value):
        self._data[key] = value
        self._data.move_to_end(key)

    def __delitem__(self, key):
        del self._data[key]

    def __iter__(self):
        return iter(self._data)

    def __len__(self):
        return len(self._data)

    def peek(self, key):
        return self._data[key]

    def __repr__(self):
        return f"{type(self).__name__}({list(self._data.items())!r})"

In [38]:
m = AccessOrderedMap()
m["a"] = 1
m["b"] = 2
m["c"] = 3

assert list(m) == ["a", "b", "c"]

_ = m["a"]
assert list(m) == ["b", "c", "a"]

# peek bypasses recency movement
assert m.peek("b") == 2
assert list(m) == ["b", "c", "a"]

print(m)

AccessOrderedMap([('b', 2), ('c', 3), ('a', 1)])


# Problem 17 — Sliding Unique Window

Process a stream and maintain only the most recent `k` **distinct** values.

When a value appears again, it becomes newest.

Example with `k=3` and stream:

`A, B, C, A, D`

states:

- `[A]`
- `[A, B]`
- `[A, B, C]`
- `[B, C, A]`
- `[C, A, D]`

## Solution 17

In [39]:
def sliding_unique_window(stream, k):
    if k <= 0:
        raise ValueError("k must be > 0")

    window = OrderedDict()

    for item in stream:
        if item in window:
            window.move_to_end(item)
        else:
            window[item] = None

        if len(window) > k:
            window.popitem(last=False)

        yield list(window)

In [40]:
states = list(sliding_unique_window(["A", "B", "C", "A", "D"], 3))

expected = [
    ["A"],
    ["A", "B"],
    ["A", "B", "C"],
    ["B", "C", "A"],
    ["C", "A", "D"],
]

assert states == expected

for state in states:
    print(state)

['A']
['A', 'B']
['A', 'B', 'C']
['B', 'C', 'A']
['C', 'A', 'D']


# Problem 18 — Compute an Order-Aware Diff

Create a function that compares two ordered mappings and reports:

- added keys,
- removed keys,
- changed values,
- moved keys.

A key is considered moved if it exists in both mappings but its index differs.

## Solution 18

In [41]:
def ordered_diff(old, new):
    old_keys = list(old)
    new_keys = list(new)

    old_set = set(old_keys)
    new_set = set(new_keys)

    added = [k for k in new_keys if k not in old_set]
    removed = [k for k in old_keys if k not in new_set]

    changed = [
        k for k in old_keys
        if k in new_set and old[k] != new[k]
    ]

    old_pos = {k: i for i, k in enumerate(old_keys)}
    new_pos = {k: i for i, k in enumerate(new_keys)}

    moved = [
        k for k in new_keys
        if k in old_set and old_pos[k] != new_pos[k]
    ]

    return {
        "added": added,
        "removed": removed,
        "changed": changed,
        "moved": moved,
    }

In [42]:
old = OrderedDict([
    ("a", 1),
    ("b", 2),
    ("c", 3),
])

new = OrderedDict([
    ("b", 20),
    ("a", 1),
    ("d", 4),
])

diff = ordered_diff(old, new)

assert diff == {
    "added": ["d"],
    "removed": ["c"],
    "changed": ["b"],
    "moved": ["b", "a"],
}

print(diff)

{'added': ['d'], 'removed': ['c'], 'changed': ['b'], 'moved': ['b', 'a']}


# Problem 19 — Reorder Using a Sequence of Commands

Given an `OrderedDict`, execute commands of the form:

- `("front", key)`
- `("back", key)`
- `("delete", key)`
- `("set", key, value)`

Rules:

- `front/back` require an existing key.
- `delete` should be idempotent.
- `set` updates in place if existing, appends if new.

Return the final mapping.

## Solution 19

In [43]:
def apply_commands(mapping, commands):
    state = OrderedDict(mapping)

    for command in commands:
        op = command[0]

        if op == "front":
            _, key = command
            state.move_to_end(key, last=False)

        elif op == "back":
            _, key = command
            state.move_to_end(key, last=True)

        elif op == "delete":
            _, key = command
            state.pop(key, None)

        elif op == "set":
            _, key, value = command
            state[key] = value

        else:
            raise ValueError(f"unknown operation: {op!r}")

    return state

In [44]:
initial = OrderedDict([("a", 1), ("b", 2), ("c", 3)])

commands = [
    ("front", "c"),
    ("set", "b", 20),
    ("set", "d", 4),
    ("back", "c"),
    ("delete", "a"),
    ("delete", "missing"),
]

result = apply_commands(initial, commands)

assert list(result.items()) == [
    ("b", 20),
    ("d", 4),
    ("c", 3),
]

print(result)

OrderedDict({'b': 20, 'd': 4, 'c': 3})


# Problem 20 — Transactional Ordered Updates

Implement a context manager-like transaction helper.

Goal:

- Work on a copy of an `OrderedDict`.
- Apply reordering and updates.
- Commit only if all validation succeeds.
- On failure, leave the original mapping unchanged.

Validation rule for this exercise:
all values must be non-negative integers.

## Solution 20

In [45]:
def validate_nonnegative_int_values(mapping):
    for key, value in mapping.items():
        if not isinstance(value, int) or isinstance(value, bool) or value < 0:
            raise ValueError(
                f"{key!r} must map to a non-negative int; got {value!r}"
            )


def transactional_update(original, operation):
    working = original.copy()

    operation(working)
    validate_nonnegative_int_values(working)

    # Commit while preserving OrderedDict identity.
    original.clear()
    original.update(working)

    return original

In [46]:
config = OrderedDict([("a", 1), ("b", 2), ("c", 3)])

def good_change(working):
    working["b"] = 20
    working.move_to_end("a")

transactional_update(config, good_change)

assert list(config.items()) == [
    ("b", 20),
    ("c", 3),
    ("a", 1),
]

snapshot = config.copy()

def bad_change(working):
    working["c"] = -999

try:
    transactional_update(config, bad_change)
except ValueError as exc:
    print("Rejected:", exc)

assert config == snapshot
assert list(config.items()) == list(snapshot.items())

print("Still valid:", config)

Rejected: 'c' must map to a non-negative int; got -999
Still valid: OrderedDict({'b': 20, 'c': 3, 'a': 1})


# Problem 21 — LRU Decorator From Scratch

Implement a small function decorator `lru_memoize(maxsize=128)`.

Assumptions:

- positional arguments are hashable,
- no keyword arguments for the base version,
- cache only successful returns,
- expose `cache_info()` and `cache_clear()`.

## Solution 21

In [47]:
@dataclass(frozen=True)
class MemoInfo:
    hits: int
    misses: int
    maxsize: int
    currsize: int


def lru_memoize(maxsize=128):
    if maxsize <= 0:
        raise ValueError("maxsize must be > 0")

    def decorator(func):
        cache = OrderedDict()
        hits = 0
        misses = 0

        def wrapper(*args):
            nonlocal hits, misses

            if args in cache:
                hits += 1
                cache.move_to_end(args)
                return cache[args]

            misses += 1
            result = func(*args)
            cache[args] = result

            if len(cache) > maxsize:
                cache.popitem(last=False)

            return result

        def cache_info():
            return MemoInfo(
                hits=hits,
                misses=misses,
                maxsize=maxsize,
                currsize=len(cache),
            )

        def cache_clear():
            nonlocal hits, misses
            cache.clear()
            hits = 0
            misses = 0

        wrapper.cache_info = cache_info
        wrapper.cache_clear = cache_clear
        wrapper._cache = cache

        return wrapper

    return decorator

In [48]:
@lru_memoize(maxsize=3)
def square(x):
    print(f"computing square({x})")
    return x * x

assert square(2) == 4
assert square(3) == 9
assert square(2) == 4  # hit
assert square(4) == 16
assert square(5) == 25  # should evict LRU entry

print(square.cache_info())
print("Cache order:", list(square._cache))

computing square(2)
computing square(3)
computing square(4)
computing square(5)
MemoInfo(hits=1, misses=4, maxsize=3, currsize=3)
Cache order: [(2,), (4,), (5,)]


### Production best practice

For normal memoization, prefer the standard-library `functools.lru_cache` or
`functools.cache`.

This exercise is valuable because it reveals the underlying recency algorithm.

# Problem 22 — Advanced Decorator Supporting Keyword Arguments

Extend the previous cache so calls with keyword arguments are supported.

Treat these calls as equivalent:

```python
f(1, x=2, y=3)
f(1, y=3, x=2)
```

Assume all arguments and keyword values are hashable.

## Solution 22

In [49]:
def make_cache_key(args, kwargs):
    # Sort keyword pairs so keyword order does not change the cache key.
    return args, tuple(sorted(kwargs.items()))


def lru_memoize_kwargs(maxsize=128):
    if maxsize <= 0:
        raise ValueError("maxsize must be > 0")

    def decorator(func):
        cache = OrderedDict()

        def wrapper(*args, **kwargs):
            key = make_cache_key(args, kwargs)

            if key in cache:
                cache.move_to_end(key)
                return cache[key]

            result = func(*args, **kwargs)
            cache[key] = result

            if len(cache) > maxsize:
                cache.popitem(last=False)

            return result

        wrapper._cache = cache
        return wrapper

    return decorator

In [50]:
calls = 0

@lru_memoize_kwargs(maxsize=4)
def combine(a, *, x=0, y=0):
    global calls
    calls += 1
    return a + x + y

assert combine(1, x=2, y=3) == 6
assert combine(1, y=3, x=2) == 6
assert calls == 1

print("Underlying calls:", calls)
print("Cache size:", len(combine._cache))

Underlying calls: 1
Cache size: 1


# Problem 23 — Top-K Most Recently Seen Unique Objects

Process `(id, payload)` records and return only the `k` most recently seen unique IDs.

The final result must be oldest -> newest among the retained IDs.

## Solution 23

In [51]:
def top_k_recent(records, k):
    if k < 0:
        raise ValueError("k must be >= 0")

    recent = OrderedDict()

    for record_id, payload in records:
        if k == 0:
            continue

        recent[record_id] = payload
        recent.move_to_end(record_id)

        if len(recent) > k:
            recent.popitem(last=False)

    return recent

In [52]:
records = [
    ("a", 1),
    ("b", 2),
    ("c", 3),
    ("a", 10),
    ("d", 4),
]

result = top_k_recent(records, 3)

# Work through the recency order explicitly:
# a -> [a]
# b -> [a, b]
# c -> [a, b, c]
# a -> [b, c, a]
# d -> [c, a, d]
assert list(result.items()) == [
    ("c", 3),
    ("a", 10),
    ("d", 4),
]

print(result)


OrderedDict({'c': 3, 'a': 10, 'd': 4})


### Reasoning check

For stateful ordered structures, explicitly trace the order after each mutation.
This prevents a common testing mistake: verifying only the final values while mentally
losing track of recency/order changes.

# Problem 24 — Design a "Second Chance" Eviction Cache

Implement a simplified second-chance cache.

Each entry stores:

```python
(value, referenced_flag)
```

Rules:

- On `get`, set `referenced_flag=True`.
- On insertion when full:
  - examine the oldest item,
  - if its flag is `False`, evict it,
  - if its flag is `True`, set the flag to `False`, move it to the end, and continue.
- Then insert the new item.

This resembles a small clock/second-chance policy but uses `OrderedDict`.

## Solution 24

In [53]:
class SecondChanceCache:
    def __init__(self, capacity):
        if capacity <= 0:
            raise ValueError("capacity must be > 0")
        self.capacity = capacity
        self._data = OrderedDict()

    def get(self, key, default=None):
        if key not in self._data:
            return default

        value, _ = self._data[key]
        self._data[key] = (value, True)
        return value

    def put(self, key, value):
        if key in self._data:
            _, flag = self._data[key]
            self._data[key] = (value, flag)
            return None

        evicted = None

        while len(self._data) >= self.capacity:
            oldest_key = next(iter(self._data))
            oldest_value, referenced = self._data[oldest_key]

            if not referenced:
                del self._data[oldest_key]
                evicted = (oldest_key, oldest_value)
                break

            self._data[oldest_key] = (oldest_value, False)
            self._data.move_to_end(oldest_key)

        self._data[key] = (value, False)
        return evicted

    def snapshot(self):
        return list(self._data.items())

In [54]:
sc = SecondChanceCache(3)
sc.put("A", 1)
sc.put("B", 2)
sc.put("C", 3)

assert sc.get("A") == 1
assert sc.get("B") == 2

evicted = sc.put("D", 4)

print("Evicted:", evicted)
print("State:", sc.snapshot())

# A and B receive a second chance; C is evicted.
assert evicted == ("C", 3)

Evicted: ('C', 3)
State: [('A', (1, False)), ('B', (2, False)), ('D', (4, False))]


# Problem 25 — Benchmark `OrderedDict` vs `deque` Fairly

The supplied material compares `OrderedDict` and `deque` for stack/queue-style use.

Improve the benchmark design.

## Requirements

Benchmark separately:

1. construction,
2. right-end popping,
3. left-end popping,
4. membership testing.

Avoid mixing construction time into pop timing.

Use fresh structures for each timed execution.

## Solution 25

In [55]:
def make_od(n):
    return OrderedDict((i, None) for i in range(n))


def make_dq(n):
    return deque(range(n))


def pop_right_od(n):
    x = make_od(n)
    while x:
        x.popitem(last=True)


def pop_left_od(n):
    x = make_od(n)
    while x:
        x.popitem(last=False)


def pop_right_dq(n):
    x = make_dq(n)
    while x:
        x.pop()


def pop_left_dq(n):
    x = make_dq(n)
    while x:
        x.popleft()


def membership_od(n, probes):
    x = make_od(n)
    return sum(p in x for p in probes)


def membership_dq(n, probes):
    x = make_dq(n)
    return sum(p in x for p in probes)

In [56]:
def best_of(statement, globals_dict, repeat_count=5, number=3):
    samples = repeat(
        statement,
        globals=globals_dict,
        repeat=repeat_count,
        number=number,
    )
    return min(samples) / number


n = 5_000
probes = [0, n // 2, n - 1, -1] * 100

benchmark_results = {
    "construct OrderedDict": best_of("make_od(n)", globals()),
    "construct deque": best_of("make_dq(n)", globals()),
    "pop right OrderedDict": best_of("pop_right_od(n)", globals()),
    "pop right deque": best_of("pop_right_dq(n)", globals()),
    "pop left OrderedDict": best_of("pop_left_od(n)", globals()),
    "pop left deque": best_of("pop_left_dq(n)", globals()),
    "membership OrderedDict": best_of(
        "membership_od(n, probes)", globals(), number=10
    ),
    "membership deque": best_of(
        "membership_dq(n, probes)", globals(), number=10
    ),
}

for name, seconds in benchmark_results.items():
    print(f"{name:28s} {seconds:.6f} s")

construct OrderedDict        0.000770 s
construct deque              0.000112 s
pop right OrderedDict        0.001762 s
pop right deque              0.000210 s
pop left OrderedDict         0.001189 s
pop left deque               0.000242 s
membership OrderedDict       0.000741 s
membership deque             0.014121 s


### Benchmark interpretation

Typical results should show:

- `deque` is highly optimized for queue/stack end operations.
- `OrderedDict` carries mapping/hash-table overhead.
- `OrderedDict` membership is usually much faster asymptotically than scanning a `deque`.
- Exact timings depend on Python version, machine, workload, key types, and benchmark size.

**Best practice:** benchmark the operation your application actually performs rather
than extrapolating from a synthetic microbenchmark.

# Problem 26 — Compare First/Last Operations Across Data Structures

Write small helper functions showing how to access the first and last key/value pair
without mutating an `OrderedDict`.

Avoid converting the entire mapping to a list.

## Solution 26

In [57]:
def first_item(mapping):
    if not mapping:
        raise KeyError("mapping is empty")
    key = next(iter(mapping))
    return key, mapping[key]


def last_item(mapping):
    if not mapping:
        raise KeyError("mapping is empty")
    key = next(reversed(mapping))
    return key, mapping[key]

In [58]:
od = OrderedDict([("a", 1), ("b", 2), ("c", 3)])

assert first_item(od) == ("a", 1)
assert last_item(od) == ("c", 3)

print(first_item(od))
print(last_item(od))

('a', 1)
('c', 3)


# Problem 27 — Safely Move a Key Only If Present

`move_to_end` raises `KeyError` for missing keys.

Implement three APIs with different semantics:

- strict: propagate `KeyError`,
- safe: return `False` when missing,
- ensure: insert a default value if missing and then move.

## Solution 27

In [59]:
def move_strict(mapping, key, *, last=True):
    mapping.move_to_end(key, last=last)


def move_safe(mapping, key, *, last=True):
    if key not in mapping:
        return False
    mapping.move_to_end(key, last=last)
    return True


def move_ensure(mapping, key, default=None, *, last=True):
    if key not in mapping:
        mapping[key] = default
    mapping.move_to_end(key, last=last)
    return mapping[key]

In [60]:
od = OrderedDict([("a", 1), ("b", 2)])

assert move_safe(od, "x") is False
assert list(od) == ["a", "b"]

assert move_ensure(od, "x", 99, last=False) == 99
assert list(od) == ["x", "a", "b"]

print(od)

OrderedDict({'x': 99, 'a': 1, 'b': 2})


# Problem 28 — Order-Sensitive Serialization

Create a simple serialization function that emits lines in mapping order:

```text
key=value
```

Then show why two mappings with identical key/value pairs can produce different serialized
text if order differs.

## Solution 28

In [61]:
def serialize_lines(mapping):
    return "\n".join(f"{key}={value}" for key, value in mapping.items())


left = OrderedDict([("host", "db"), ("port", 5432)])
right = OrderedDict([("port", 5432), ("host", "db")])

assert dict(left) == dict(right)
assert serialize_lines(left) != serialize_lines(right)

print("LEFT")
print(serialize_lines(left))
print()
print("RIGHT")
print(serialize_lines(right))

LEFT
host=db
port=5432

RIGHT
port=5432
host=db


### Best practice

When order affects serialization, hashing, signatures, migrations, or execution,
make that requirement explicit in your data model and tests.

# Problem 29 — Reverse Iteration Without Copying

Create a function that yields `(key, value)` pairs from newest to oldest without building
a reversed list.

## Solution 29

In [62]:
def reversed_items(mapping):
    for key in reversed(mapping):
        yield key, mapping[key]

In [63]:
od = OrderedDict([("a", 1), ("b", 2), ("c", 3)])

assert list(reversed_items(od)) == [
    ("c", 3),
    ("b", 2),
    ("a", 1),
]

for item in reversed_items(od):
    print(item)

('c', 3)
('b', 2)
('a', 1)


# Problem 30 — Stress-Test LRU Invariants

Use randomized operations to test an LRU cache against a simple reference model.

Invariant:

- keys in the implementation and model must match,
- values must match,
- recency order must match,
- size must never exceed capacity.

## Solution 30

In [64]:
def reference_lru_put(model, capacity, key, value):
    # model is a list of [key, value] from LRU -> MRU
    for i, (k, _) in enumerate(model):
        if k == key:
            model.pop(i)
            model.append([key, value])
            return None

    model.append([key, value])

    if len(model) > capacity:
        return tuple(model.pop(0))

    return None


def reference_lru_get(model, key, default=None):
    for i, (k, value) in enumerate(model):
        if k == key:
            model.pop(i)
            model.append([k, value])
            return value

    return default

In [65]:
rng = random.Random(42)
capacity = 5
cache = LRUCache(capacity)
model = []

for step in range(5_000):
    key = rng.randrange(10)

    if rng.random() < 0.6:
        value = rng.randrange(1_000)
        got_eviction = cache.put(key, value)
        expected_eviction = reference_lru_put(model, capacity, key, value)
        assert got_eviction == expected_eviction
    else:
        got = cache.get(key, default=None)
        expected = reference_lru_get(model, key, default=None)
        assert got == expected

    assert len(cache) <= capacity
    assert cache.items_lru_to_mru() == [tuple(x) for x in model]

print("5,000 randomized operations passed.")

5,000 randomized operations passed.


# Bonus Challenge 31 — Segmented LRU (SLRU)

A segmented LRU cache has two regions:

- **probationary**: newly inserted entries,
- **protected**: entries that have been accessed again.

Rules for this simplified version:

- New keys enter probationary.
- A probationary hit promotes the key to protected.
- A protected hit moves it to the protected MRU end.
- If protected exceeds its limit, demote its LRU key back to probationary.
- If total capacity is exceeded, evict probationary LRU first.

Implement a compact version.

## Solution 31

In [66]:
class SegmentedLRU:
    def __init__(self, probationary_capacity, protected_capacity):
        if probationary_capacity <= 0 or protected_capacity <= 0:
            raise ValueError("segment capacities must be > 0")

        self.probationary_capacity = probationary_capacity
        self.protected_capacity = protected_capacity
        self.probationary = OrderedDict()
        self.protected = OrderedDict()

    @property
    def total_capacity(self):
        return self.probationary_capacity + self.protected_capacity

    def _rebalance_protected(self):
        while len(self.protected) > self.protected_capacity:
            key, value = self.protected.popitem(last=False)
            self.probationary[key] = value

    def _rebalance_total(self):
        while len(self.probationary) + len(self.protected) > self.total_capacity:
            if self.probationary:
                self.probationary.popitem(last=False)
            else:
                self.protected.popitem(last=False)

    def put(self, key, value):
        if key in self.protected:
            self.protected[key] = value
            self.protected.move_to_end(key)
            return

        if key in self.probationary:
            self.probationary[key] = value
            return

        self.probationary[key] = value
        self._rebalance_total()

    def get(self, key, default=None):
        if key in self.protected:
            value = self.protected[key]
            self.protected.move_to_end(key)
            return value

        if key in self.probationary:
            value = self.probationary.pop(key)
            self.protected[key] = value
            self._rebalance_protected()
            self._rebalance_total()
            return value

        return default

    def snapshot(self):
        return {
            "probationary": list(self.probationary.items()),
            "protected": list(self.protected.items()),
        }

In [67]:
slru = SegmentedLRU(probationary_capacity=2, protected_capacity=2)

slru.put("A", 1)
slru.put("B", 2)
slru.put("C", 3)
slru.put("D", 4)

print("Initial:", slru.snapshot())

# Promote some entries
print("get C:", slru.get("C"))
print("get D:", slru.get("D"))

slru.put("E", 5)
print("After promotions + E:", slru.snapshot())

Initial: {'probationary': [('A', 1), ('B', 2), ('C', 3), ('D', 4)], 'protected': []}
get C: 3
get D: 4
After promotions + E: {'probationary': [('B', 2), ('E', 5)], 'protected': [('C', 3), ('D', 4)]}


# Bonus Challenge 32 — Ordered Dependency Scheduler

Given tasks and dependencies, perform a deterministic topological sort that preserves the
original task order whenever multiple tasks are simultaneously ready.

Use an `OrderedDict` for the ready queue.

Input:

```python
tasks = ["lint", "test", "build", "deploy"]
dependencies = {
    "lint": set(),
    "test": set(),
    "build": {"lint", "test"},
    "deploy": {"build"},
}
```

## Solution 32

In [68]:
def stable_topological_sort(tasks, dependencies):
    tasks = list(tasks)
    task_set = set(tasks)

    deps = {
        task: set(dependencies.get(task, ()))
        for task in tasks
    }

    unknown = {
        dep
        for task_deps in deps.values()
        for dep in task_deps
        if dep not in task_set
    }
    if unknown:
        raise KeyError(f"unknown dependencies: {sorted(unknown)!r}")

    ready = OrderedDict(
        (task, None)
        for task in tasks
        if not deps[task]
    )

    result = []
    emitted = set()

    while ready:
        task, _ = ready.popitem(last=False)

        if task in emitted:
            continue

        emitted.add(task)
        result.append(task)

        for candidate in tasks:
            if candidate in emitted:
                continue

            deps[candidate].discard(task)

            if not deps[candidate] and candidate not in ready:
                ready[candidate] = None

    if len(result) != len(tasks):
        unresolved = [task for task in tasks if task not in emitted]
        raise ValueError(f"cycle detected involving: {unresolved!r}")

    return result

In [69]:
tasks = ["lint", "test", "build", "deploy"]

dependencies = {
    "lint": set(),
    "test": set(),
    "build": {"lint", "test"},
    "deploy": {"build"},
}

order = stable_topological_sort(tasks, dependencies)

assert order == ["lint", "test", "build", "deploy"]
print(order)

['lint', 'test', 'build', 'deploy']


# Bonus Challenge 33 — Invariant Checker for Ordered Structures

Write a reusable validator for an LRU-style structure.

It should verify:

- the number of keys equals the number of unique keys,
- size <= capacity,
- expected keys and values match,
- expected order matches exactly.

Return a descriptive list of errors instead of raising immediately.

## Solution 33

In [70]:
def validate_lru_state(cache, expected_items):
    errors = []
    actual = cache.items_lru_to_mru()

    keys = [k for k, _ in actual]

    if len(keys) != len(set(keys)):
        errors.append("duplicate keys detected")

    if len(actual) > cache.capacity:
        errors.append(
            f"capacity exceeded: size={len(actual)}, capacity={cache.capacity}"
        )

    if dict(actual) != dict(expected_items):
        errors.append(
            f"mapping mismatch: actual={dict(actual)!r}, "
            f"expected={dict(expected_items)!r}"
        )

    if actual != list(expected_items):
        errors.append(
            f"order mismatch: actual={actual!r}, expected={list(expected_items)!r}"
        )

    return errors

In [71]:
cache = LRUCache(3)
cache.put("a", 1)
cache.put("b", 2)
cache.put("c", 3)
cache.get("a")

errors = validate_lru_state(
    cache,
    [("b", 2), ("c", 3), ("a", 1)],
)

assert errors == []
print("Validation errors:", errors)

Validation errors: []


# Bonus Challenge 34 — When NOT to Use `OrderedDict`

Choose the most appropriate structure for each use case:

1. Need insertion order only.
2. Need FIFO queue with very frequent left pops.
3. Need LRU recency updates and keyed lookup.
4. Need unique values with no meaningful order.
5. Need priority by numeric score.
6. Need order-sensitive mapping equality.

## Solution 34

1. **Insertion order only:** use a regular `dict` in modern Python.
2. **FIFO queue:** use `collections.deque`.
3. **LRU recency + keyed lookup:** `OrderedDict` is a strong low-level fit
   (or use a standard/library cache abstraction when available).
4. **Unique unordered values:** use `set`.
5. **Priority by numeric score:** usually use `heapq` or a purpose-built priority queue.
6. **Order-sensitive mapping equality:** `OrderedDict` provides this semantic directly
   when compared with another `OrderedDict`.

The best data structure is not the one with the most features; it is the one whose
semantics and complexity match the workload.

# Mini Drills — 20 Short Problems

Try these without looking up the answers.

1. Move key `"x"` to the front.
2. Remove and return the oldest pair.
3. Remove and return the newest pair.
4. Read the newest key without mutation.
5. Read the oldest key without mutation.
6. Check whether two ordered mappings have the same sequence of pairs.
7. Update a value without changing its existing position.
8. Update a value and move the key to the end.
9. Add a new key at the front.
10. Reverse-iterate keys.
11. Build an ordered set from a list.
12. Deduplicate with last occurrence deciding order.
13. Make a successful read update LRU order.
14. Peek without updating LRU order.
15. Evict LRU.
16. Evict MRU.
17. Safely move a key only if it exists.
18. Pop all items from newest to oldest.
19. Pop all items from oldest to newest.
20. Compare mapping equality independently of order.

## Mini Drill Solutions

In [72]:
# Assume this starting mapping for examples:
od = OrderedDict([("a", 1), ("x", 2), ("z", 3)])

# 1
od.move_to_end("x", last=False)

# 2
oldest = od.popitem(last=False)

# reset
od = OrderedDict([("a", 1), ("x", 2), ("z", 3)])

# 3
newest = od.popitem(last=True)

# reset
od = OrderedDict([("a", 1), ("x", 2), ("z", 3)])

# 4
newest_key = next(reversed(od))

# 5
oldest_key = next(iter(od))

# 6
same_sequence = list(od1.items()) == list(od2.items()) if False else "example"

# 7
od["x"] = 200  # existing key keeps its position

# 8
od["x"] = 201
od.move_to_end("x")

# 9
od["new"] = 999
od.move_to_end("new", last=False)

# 10
reverse_keys = list(reversed(od))

# 11
ordered_set = OrderedDict.fromkeys(["a", "b", "a", "c"])

# 12
def dedupe_last(items):
    out = OrderedDict()
    for item in items:
        out[item] = None
        out.move_to_end(item)
    return list(out)

# 13
value = od["x"]
od.move_to_end("x")

# 14
value = od.get("x")

# 15
lru_pair = od.popitem(last=False)

# reset
od = OrderedDict([("a", 1), ("x", 2), ("z", 3)])

# 16
mru_pair = od.popitem(last=True)

# reset
od = OrderedDict([("a", 1), ("x", 2), ("z", 3)])

# 17
if "x" in od:
    od.move_to_end("x")

# 18
copy1 = od.copy()
newest_to_oldest = []
while copy1:
    newest_to_oldest.append(copy1.popitem(last=True))

# 19
copy2 = od.copy()
oldest_to_newest = []
while copy2:
    oldest_to_newest.append(copy2.popitem(last=False))

# 20
left = OrderedDict([("a", 1), ("b", 2)])
right = OrderedDict([("b", 2), ("a", 1)])
same_mapping = dict(left) == dict(right)

print("Mini drills executed successfully.")

Mini drills executed successfully.


# Final Best-Practice Checklist

- Prefer plain `dict` when you only need insertion order.
- Prefer `OrderedDict` when reordering existing keys is a first-class operation.
- Use `move_to_end(key)` for MRU-style movement.
- Use `move_to_end(key, last=False)` for move-to-front behavior.
- Use `popitem(last=False)` for oldest/LRU/FIFO-style eviction.
- Use `popitem(last=True)` for newest/MRU/LIFO-style eviction.
- Remember that missing keys passed to `move_to_end` raise `KeyError`.
- Be explicit about whether reads update recency.
- Be explicit about whether updates change order.
- Test order invariants, not only key/value equality.
- Do not use sets in algorithms where sequence order matters unless the set is only for
  membership bookkeeping.
- Prefer `deque` for pure queue/stack workloads.
- Benchmark realistic workloads before choosing a structure for performance reasons.
- Prefer standard-library abstractions such as `functools.lru_cache` when they already
  solve the production problem.
- Document eviction semantics: LRU, MRU, FIFO, latest-update, second-chance, etc.
- Validate capacities and other constructor invariants early.
- Keep mutation logic centralized so ordering rules remain consistent.
- Add randomized or property-style tests for stateful recency algorithms.

# Suggested Further Exercises

1. Add thread safety to `LRUCache` using a lock.
2. Add per-key TTL values to `TTLLRUCache`.
3. Add a callback invoked whenever an item is evicted.
4. Track eviction reasons: capacity vs expiration.
5. Implement a weighted LRU where entries consume different capacities.
6. Implement a two-level cache with hot/cold segments.
7. Add persistence by serializing ordered cache state.
8. Implement snapshot/restore while preserving recency.
9. Compare memory usage of `dict`, `OrderedDict`, and `deque`.
10. Reimplement one solution using plain `dict` and document what becomes awkward.